In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import  re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


In [48]:
df=pd.read_csv('P:\AI-Resume-Screener\data\dataset_cleaned.csv')

In [49]:
df.head()

,Role,Resume,decision,Job_Description
0,E-commerce Specialist,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...
1,Game Developer,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...
2,Human Resources Specialist,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...
3,E-commerce Specialist,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...
4,E-commerce Specialist,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...


In [ ]:
label=LabelEncoder()

df['Role'] = label.fit_transform(df['Role'])
df.head()

,Role,Resume,decision,Job_Description
0,17,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...
1,19,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...
2,22,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...
3,17,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...
4,17,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...


In [51]:
df.notna().sum()

Role               10174
Resume             10174
decision           10174
Job_Description    10174
dtype: int64

In [52]:
df.isnull().sum()  

Role               0
Resume             0
decision           0
Job_Description    0
dtype: int64

In [87]:
word_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000)
combined = pd.concat([
    df["Resume"],
    df["Job_Description"]
])
word_vectorizer.fit(combined)
resume_vectorized=word_vectorizer.transform(df["Resume"])
job_description_vectorized=word_vectorizer.transform(df["Job_Description"])

In [88]:

similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    similarity.append(score)

In [89]:
df['similarity'] = similarity
df

,Role,Resume,decision,Job_Description,similarity,resume_length,jd_length,keyword_overlap_ratio
0,17,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.142312,253,11,0.272727
1,19,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.112280,40,11,0.181818
2,22,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.154562,286,13,0.384615
3,17,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.106139,234,11,0.272727
4,17,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.047507,271,12,0.250000
...,...,...,...,...,...,...,...,...
10169,27,sample resume diana miller diana miller contac...,reject,comprehensive job description product manager ...,0.488060,310,406,0.147783
10170,35,sample resume grace taylor grace taylor contac...,reject,sample job description ui engineer job title u...,0.463782,319,378,0.211640
10171,35,sample resume hank brown hank brown ui enginee...,select,job description ui engineer role job title ui ...,0.439262,343,329,0.191489
10172,12,sample resume diana wilson diana wilson contac...,reject,comprehensive job description data engineer ro...,0.510677,317,366,0.174863


In [90]:
df["resume_length"] = df["Resume"].apply(lambda x: len(x.split()))
df["jd_length"] = df["Job_Description"].apply(lambda x: len(x.split()))

In [98]:
keyword_overlap_ratio=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    resume_words=df['Resume'][i].split(" ")
    resume_words = {
        w for w in resume_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=set(job_words).intersection(set(resume_words))
    keyword_overlap_ratio.append(len(comman)/len(job_words))

df['keyword_overlap_ratio'] = keyword_overlap_ratio

In [99]:
df.head()

,Role,Resume,decision,Job_Description,similarity,resume_length,jd_length,keyword_overlap_ratio
0,17,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.142312,253,11,0.272727
1,19,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.112280,40,11,0.181818
2,22,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.154562,286,13,0.384615
3,17,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.106139,234,11,0.272727
4,17,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.047507,271,12,0.250000


In [100]:
encoder=LabelEncoder()
X=df[['similarity','resume_length','jd_length','keyword_overlap_ratio']]
y=encoder.fit_transform(df["decision"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [101]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    random_state=42
)

param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300, 500],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

In [102]:


pred = best_model.predict(X_test)

print(accuracy_score(y_test,pred))

print(classification_report(y_test,pred))

print(confusion_matrix(y_test,pred))

0.5283327874222077
              precision    recall  f1-score   support

           0       0.54      0.50      0.52      1548
           1       0.52      0.55      0.54      1505

    accuracy                           0.53      3053
   macro avg       0.53      0.53      0.53      3053
weighted avg       0.53      0.53      0.53      3053

[[779 769]
 [671 834]]
